# 07 Linear models

OLS benchmarks (ols3, ols_pars, ols_full) and penalised regressions (ridge, lasso, enet, elanet) on the complete-case and imputed samples, with rolling windows of 8 training, 2 validation and 1 test year over 2010–2025. The two samples are also compared on the same observations and on the observations recovered by imputation. The main run uses the `_mv25` samples; the unrestricted samples are an extension. Sections 3.6.1 and 3.7, Chapter 4.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, gc
import numpy as np
import pandas as pd

EST_DIR   = "/content/drive/MyDrive/Thesis/data"   # estimation samples (notebook 06)
PANEL_DIR = "/content/drive/MyDrive/Thesis/data"   # panel_characteristics.parquet (notebook 05)
OUT_DIR   = "/content/drive/MyDrive/Thesis/data"   # results of this notebook

SAMPLES = ["complete_case_mv25", "imputed_mv25"]   # main samples, above EUR 25 mn;
# complete-case first, imputed second. Extension without the threshold:
# SAMPLES = ["complete_case", "imputed"]
FIRST_TEST, LAST_TEST = 2010, 2025
N_TRAIN, N_VAL = 8, 2
ALPHAS = np.logspace(-6, -1, 21)          # lasso and elastic net
ALPHAS_RIDGE = np.logspace(-2, 8, 41)     # ridge

## Functions

In [ ]:
import itertools
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, enet_path

CHARS = ["size", "bm", "operatingprofitability", "totalassetgrowth", "ret_2_12",
         "issues_1_36", "accrualschange", "roa", "investment", "ret_1",
         "ret_12_36", "dy", "beta", "vola", "turnover", "debttoprice",
         "salestoprice", "cftoprice", "earningstoprice", "issues_1_12",
         "fcdispersion", "grossprofitability"]

OLS3 = ["size", "bm", "ret_2_12"]          # Gu, Kelly and Xiu (2020)


# Predictor sets

def build_predictor_sets(df, chars=CHARS):
    """
    The two nested predictor sets of Drobetz and Otto (2021).

    Parsimonious: the 22 rank-transformed characteristics and indicators for
    country and industry (they use ISO country codes and two-digit SIC codes;
    here the four countries and the ICB sectors). Full: the parsimonious set
    plus all two-way interactions between characteristics and their second-
    and third-order powers. Indicators are built once on the whole sample, so
    every window shares the same columns; one category of each is dropped.
    """
    C = df[chars].to_numpy(dtype=np.float32)
    dummies = pd.concat([
        pd.get_dummies(df["country"], prefix="cty", drop_first=True, dtype=np.float32),
        pd.get_dummies(df["sector"], prefix="sec", drop_first=True, dtype=np.float32)],
        axis=1)

    pairs = list(itertools.combinations(range(len(chars)), 2))
    inter = np.column_stack([C[:, i] * C[:, j] for i, j in pairs]).astype(np.float32)
    names_inter = [f"{chars[i]}*{chars[j]}" for i, j in pairs]

    pars = np.hstack([C, dummies.to_numpy()])
    names_pars = list(chars) + list(dummies.columns)
    full = np.hstack([C, inter, C ** 2, C ** 3, dummies.to_numpy()])
    names_full = (list(chars) + names_inter + [f"{c}^2" for c in chars]
                  + [f"{c}^3" for c in chars] + list(dummies.columns))
    return pars, names_pars, full, names_full


def rolling_splits(dates, first_test_year, last_test_year, n_train=8, n_val=2):
    """Rolling windows of fixed length, refitted once a year (by predictor month)."""
    years = pd.DatetimeIndex(dates).year
    for y in range(first_test_year, last_test_year + 1):
        tr = (years >= y - n_val - n_train) & (years <= y - n_val - 1)
        va = (years >= y - n_val) & (years <= y - 1)
        te = years == y
        if te.sum() and tr.sum() and va.sum():
            yield y, np.asarray(tr), np.asarray(va), np.asarray(te)


# Models

def fit_ols(X, y):
    return LinearRegression().fit(X, y)


def fit_penalised(Xtr, ytr, Xva, yva, alphas, alphas_ridge):
    """
    Ridge, lasso and elastic net (mixing parameter 0.5), each tuned on the
    validation sample; the elastic net of Drobetz and Otto is whichever of the
    three attains the lowest validation error, their mixing parameter being
    chosen from {0, 0.5, 1}.

    Training data are centred and the cross-product matrix computed once, from
    which the full penalty path of each model follows; this keeps estimation
    fast with several hundred predictors. Forecasting models are those
    estimated on the training sample, as in Drobetz and Otto.
    """
    xm = Xtr.mean(axis=0, dtype=np.float64)
    ym = float(ytr.mean())
    Xc = (Xtr - xm).astype(np.float64)
    yc = ytr - ym
    G = Xc.T @ Xc
    Xy = Xc.T @ yc
    Xva_c = (Xva - xm).astype(np.float64)
    res = {}

    for name, l1 in [("lasso", 1.0), ("enet", 0.5)]:
        a_path, coefs, _ = enet_path(Xc, yc, l1_ratio=l1, alphas=alphas,
                                     precompute=G, Xy=Xy, max_iter=20_000, tol=1e-6)
        mse = ((yva[:, None] - (Xva_c @ coefs + ym)) ** 2).mean(axis=0)
        k = int(np.argmin(mse))
        res[name] = {"alpha": float(a_path[k]), "coef": coefs[:, k], "val_mse": float(mse[k]),
                     "at_boundary": k in (0, len(a_path) - 1)}

    evals, V = np.linalg.eigh(G)
    VtXy = V.T @ Xy
    best = None
    for k, a in enumerate(alphas_ridge):
        coef = V @ (VtXy / (evals + a))
        mse = float(np.mean((yva - (Xva_c @ coef + ym)) ** 2))
        if best is None or mse < best["val_mse"]:
            best = {"alpha": float(a), "coef": coef, "val_mse": mse,
                    "at_boundary": k in (0, len(alphas_ridge) - 1)}
    res["ridge"] = best

    for r in res.values():
        r["intercept"] = ym - float(xm @ r["coef"])
    sel = min(("ridge", "enet", "lasso"), key=lambda m: res[m]["val_mse"])
    res["elanet"] = {**res[sel], "rho": {"ridge": 0.0, "enet": 0.5, "lasso": 1.0}[sel]}
    return res


def predict_linear(X, r):
    return X.astype(np.float64) @ r["coef"] + r["intercept"]


# Evaluation

def r2_oos(y, yhat):
    """Out-of-sample R^2 against a forecast of zero (Gu, Kelly and Xiu, 2020)."""
    y, yhat = np.asarray(y), np.asarray(yhat)
    return 1.0 - np.sum((y - yhat) ** 2) / np.sum(y ** 2)


def r2_cross_section(df, m, y="target"):
    """R^2 after removing each month's average: ability to rank, net of the market level."""
    yy = df[y] - df.groupby("date")[y].transform("mean")
    ff = df[m] - df.groupby("date")[m].transform("mean")
    return 1.0 - ((yy - ff) ** 2).sum() / (yy ** 2).sum()


def predictive_slope(y, yhat):
    """Cov(realised, predicted) / Var(predicted): 1 means unbiased forecast dispersion."""
    yhat = np.asarray(yhat)
    v = np.var(yhat)
    return np.cov(np.asarray(y), yhat, bias=True)[0, 1] / v if v > 0 else np.nan


def newey_west_se(x, lags=None):
    x = np.asarray(x, dtype=float)
    T = len(x)
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    u = x - x.mean()
    s = np.mean(u ** 2)
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * np.mean(u[l:] * u[:-l])
    return np.sqrt(s / T)


def diebold_mariano(df, model_a, model_b, y="target"):
    """Modified Diebold-Mariano statistic; positive: model_b more accurate than model_a."""
    d = ((df[y] - df[model_a]) ** 2 - (df[y] - df[model_b]) ** 2).groupby(df["date"]).mean()
    return d.mean() / newey_west_se(d.values)


def decile_portfolios(df, pred, ret="raw_target", weight="MV"):
    """Monthly decile portfolios on the forecast, equal- and value-weighted."""
    d = df.dropna(subset=[pred, ret]).copy()
    d["dec"] = d.groupby("date")[pred].transform(
        lambda s: pd.qcut(s.rank(method="first"), 10, labels=False))
    ew = d.groupby(["date", "dec"])[ret].mean().unstack()
    d["_w"] = d[weight].clip(lower=0)
    d["_wr"] = d["_w"] * d[ret]
    g = d.groupby(["date", "dec"])
    vw = (g["_wr"].sum(min_count=1) / g["_w"].sum(min_count=1)).unstack()
    return ew, vw


def portfolio_stats(port):
    hl = (port[9] - port[0]).dropna()
    return {"mean_%": 100 * hl.mean(), "t_stat": hl.mean() / newey_west_se(hl.values),
            "sharpe_ann": hl.mean() / hl.std() * np.sqrt(12), "months": len(hl)}


In [ ]:
PEN = ["ridge", "lasso", "enet", "elanet"]
MODELS = ["ols3", "ols_pars", "ols_full", "ridge", "lasso", "enet", "elanet"]
DO_R2 = {"ols_full": -0.52, "ols_pars": 0.57, "elanet": 1.21}
DO_R2_LARGE = {"ols_full": -1.16, "ols_pars": 0.11, "elanet": 0.75}
DO_R2_SMALL = {"ols_full": -0.10, "ols_pars": 0.90, "elanet": 1.55}
DO_PS = {"ols_full": 0.45, "ols_pars": 0.65, "elanet": 0.92}


def load_sample(name, est_dir, panel_dir):
    est = pd.read_parquet(os.path.join(est_dir, f"estimation_{name}.parquet"))
    est["date"] = pd.to_datetime(est["date"])
    raw = pd.read_parquet(os.path.join(panel_dir, "panel_characteristics.parquet"),
                          columns=["symbol", "date", "target"])
    raw["date"] = pd.to_datetime(raw["date"])
    return (est.merge(raw.rename(columns={"target": "raw_target"}), on=["symbol", "date"],
                      how="left").sort_values(["date", "symbol"]).reset_index(drop=True))


def estimate(est, first_test, last_test, n_train, n_val, alphas, alphas_ridge, label=""):
    """All linear models over the rolling windows; returns forecasts and tuning records."""
    X_pars, names_pars, X_full, names_full = build_predictor_sets(est)
    y = est["target"].to_numpy()
    print(f"[{label}] {len(est):,} obs | parsimonious {X_pars.shape[1]}, full {X_full.shape[1]} predictors")
    preds, tuning = [], []
    for yr, tr, va, te in rolling_splits(est["date"], first_test, last_test, n_train, n_val):
        out = est.loc[te, ["symbol", "date", "target", "raw_target", "MV", "country"]].copy()
        out["ols3"] = fit_ols(est.loc[tr, OLS3], y[tr]).predict(est.loc[te, OLS3])
        out["ols_pars"] = fit_ols(X_pars[tr], y[tr]).predict(X_pars[te])
        out["ols_full"] = fit_ols(X_full[tr], y[tr]).predict(X_full[te])
        res = fit_penalised(X_full[tr], y[tr], X_full[va], y[va], alphas, alphas_ridge)
        for m in PEN:
            out[m] = predict_linear(X_full[te], res[m])
            tuning.append({"model": m, "year": yr, "alpha": res[m]["alpha"],
                           "rho": res[m].get("rho", np.nan),
                           "nonzero": int(np.sum(np.abs(res[m]["coef"]) > 1e-12)),
                           "at_boundary": res[m]["at_boundary"], "coef": res[m]["coef"]})
        preds.append(out)
        print(f"   {yr}: elanet rho {res['elanet']['rho']:.1f}, "
              f"{np.sum(np.abs(res['elanet']['coef']) > 1e-12):>3} non-zero", end="\n" if yr % 4 == 1 else " |")
    print()
    if "n_imputed" in est.columns:
        P = pd.concat(preds, ignore_index=True)
        P = P.merge(est[["symbol", "date", "n_imputed"]], on=["symbol", "date"], how="left")
    else:
        P = pd.concat(preds, ignore_index=True)
    return P, pd.DataFrame(tuning), names_full


def annual_r2(P, models, y="target"):
    return pd.DataFrame({m: P.groupby(P["date"].dt.year).apply(
        lambda g: 100 * r2_oos(g[y], g[m]), include_groups=False) for m in models})


def periods_of(P):
    yr = P["date"].dt.year
    return {"2010-2025": yr >= 2010, "2010-2020": yr <= 2020, "2021-2025": yr >= 2021}


def avg_annual_table(P, models, y="target"):
    a = annual_r2(P, models, y)
    return pd.DataFrame({"2010-2025": a.mean(), "2010-2020": a.loc[:2020].mean(),
                         "2021-2025": a.loc[2021:].mean()})


def side_by_side(tables, ref=None, ref_label="DO"):
    out = pd.concat(tables, axis=1)
    if ref is not None:
        out[(ref_label, "2000-2020")] = pd.Series(ref)
    return out


## Estimation

Each sample is loaded, its predictor sets built, and all models estimated over the
sixteen rolling windows. The predictor matrices are released before the next sample.

In [ ]:
R = {}
t0 = time.time()
for s in SAMPLES:
    est = load_sample(s, EST_DIR, PANEL_DIR)
    print(f"\n{s}: {len(est):,} security-months, {est['symbol'].nunique():,} securities, "
          f"unwinsorised target matched {est['raw_target'].notna().mean():.1%}")
    P, T, NAMES_FULL = estimate(est, FIRST_TEST, LAST_TEST, N_TRAIN, N_VAL,
                                ALPHAS, ALPHAS_RIDGE, label=s)
    R[s] = {"P": P, "T": T, "names": NAMES_FULL}
    del est; gc.collect()
print(f"\nDone in {time.time()-t0:.0f}s.")
for s in SAMPLES:
    print(f"  {s:14s} {len(R[s]['P']):>9,} out-of-sample forecasts")

## Each sample on its own observations

### Out-of-sample $R^2$

Average of annual $R^2$, as reported by Drobetz and Otto, then the pooled version.

In [ ]:
avg = side_by_side({s: avg_annual_table(R[s]["P"], MODELS) for s in SAMPLES}, DO_R2)
print("Average of annual R^2 (%):")
print(avg.round(3).to_string())

pooled = pd.concat({s: pd.DataFrame({p: {m: 100 * r2_oos(R[s]["P"].loc[mk, "target"], R[s]["P"].loc[mk, m])
                                         for m in MODELS} for p, mk in periods_of(R[s]["P"]).items()})
                    for s in SAMPLES}, axis=1)
print("\nPooled R^2 (%):")
print(pooled.round(3).to_string())

In [ ]:
for s in SAMPLES:
    print(f"\nR^2 (%) by test year -- {s}:")
    print(annual_r2(R[s]["P"], MODELS).round(2).to_string())

### Ranking stocks versus forecasting the market

Cross-sectional $R^2$, net of each month's average, and the correlation between
annual $R^2$ and the market's average excess return in that year.

In [ ]:
cs = pd.concat({s: pd.DataFrame({p: {m: 100 * r2_cross_section(R[s]["P"][mk], m) for m in MODELS}
                                  for p, mk in periods_of(R[s]["P"]).items()}) for s in SAMPLES}, axis=1)
print("Cross-sectional R^2 (%):")
print(cs.round(3).to_string())

corr = {}
for s in SAMPLES:
    P = R[s]["P"]
    mkt = P.groupby("date")["target"].mean().groupby(lambda d: d.year).mean()
    corr[s] = annual_r2(P, MODELS).corrwith(mkt)
print("\nCorrelation of annual R^2 with the year's average excess return:")
print(pd.DataFrame(corr).round(2).to_string())

### Large and small firms

Split each month at the median market capitalisation of the sample.

In [ ]:
size_tabs = {}
for s in SAMPLES:
    P = R[s]["P"]
    large = P["MV"] >= P.groupby("date")["MV"].transform("median")
    size_tabs[s] = pd.DataFrame({"large": annual_r2(P[large], MODELS).mean(),
                                 "small": annual_r2(P[~large & P["MV"].notna()], MODELS).mean()})
size_r2 = pd.concat(size_tabs, axis=1)
size_r2[("DO", "large")] = pd.Series(DO_R2_LARGE)
size_r2[("DO", "small")] = pd.Series(DO_R2_SMALL)
print("Average of annual R^2 (%) by size, 2010-2025:")
print(size_r2.round(3).to_string())

### Predictive slopes

$\mathrm{Cov}(r,\hat r)/\mathrm{Var}(\hat r)$ averaged over test years; below one,
forecasts are too dispersed. When a model's forecasts are nearly
constant in a year, as with heavily penalised ridge, the ratio becomes unstable; such
years are flagged, since a single one can dominate the average.

In [ ]:
ps = {}
for s in SAMPLES:
    P = R[s]["P"]
    a = pd.DataFrame({m: P.groupby(P["date"].dt.year).apply(
        lambda g: predictive_slope(g["target"], g[m]), include_groups=False) for m in MODELS})
    ps[s] = pd.DataFrame({"2010-2025": a.mean(), "2010-2020": a.loc[:2020].mean(),
                          "2021-2025": a.loc[2021:].mean()})
    unstable = a.stack()[a.stack().abs() > 5]
    if len(unstable):
        print(f"{s}: unstable slopes (|PS| > 5, near-constant forecasts): "
              + ", ".join(f"{m} {y}: {v:.1f}" for (y, m), v in unstable.items()))
ps_tab = side_by_side(ps, DO_PS)
print(ps_tab.round(3).to_string())

### Diebold-Mariano tests within each sample

Positive: the column model is more accurate than the row model; |DM| > 1.96 is
significant at 5%.

In [ ]:
dm = {}
for s in SAMPLES:
    d = pd.DataFrame(index=MODELS, columns=MODELS, dtype=float)
    for a in MODELS:
        for b in MODELS:
            if a != b:
                d.loc[a, b] = diebold_mariano(R[s]["P"], a, b)
    dm[s] = d
    print(f"\n{s}:")
    print(d.round(2).to_string())

## Comparing the samples

### On the same observations

The complete-case security-months, forecast by the model estimated on each sample and
evaluated against the same target. A positive Diebold-Mariano statistic means that
the model trained on the imputed sample forecasts these firms more accurately.

In [ ]:
# SAMPLES[0] is the complete-case sample, SAMPLES[1] the imputed one
Pc, Pi = R[SAMPLES[0]]["P"], R[SAMPLES[1]]["P"]
common = Pc[["symbol", "date", "target", "raw_target", "MV"] + MODELS].merge(
    Pi[["symbol", "date"] + MODELS].rename(columns={m: f"{m}_imp" for m in MODELS}),
    on=["symbol", "date"], how="inner")
print(f"Common observations: {len(common):,} of {len(Pc):,} complete-case forecasts "
      f"({100*len(common)/len(Pc):.1f}%)\n")

rows = []
for m in MODELS:
    a_cc = annual_r2(common, [m]).iloc[:, 0]
    a_im = annual_r2(common.rename(columns={f"{m}_imp": "x"}), ["x"]).iloc[:, 0]
    rows.append({"model": m,
                 "R2 trained CC": a_cc.mean(), "R2 trained imputed": a_im.mean(),
                 "CS R2 trained CC": 100 * r2_cross_section(common, m),
                 "CS R2 trained imputed": 100 * r2_cross_section(common, f"{m}_imp"),
                 "DM (imputed better > 0)": diebold_mariano(common, m, f"{m}_imp")})
same_obs = pd.DataFrame(rows).set_index("model")
print("Average of annual R^2 and cross-sectional R^2 (%), 2010-2025, same observations:")
print(same_obs.round(3).to_string())

yr = common["date"].dt.year
sub = {}
for lab, mk in [("2010-2020", yr <= 2020), ("2021-2025", yr >= 2021)]:
    sub[lab] = pd.Series({m: diebold_mariano(common[mk], m, f"{m}_imp") for m in MODELS})
print("\nDiebold-Mariano by sub-period (imputed better > 0):")
print(pd.DataFrame(sub).round(2).to_string())

### On the recovered observations

Security-months present only in the imputed sample: the firms the complete-case
restriction discards, forecast by the models estimated on the imputed sample.

In [ ]:
key_c = pd.MultiIndex.from_frame(Pc[["symbol", "date"]])
key_i = pd.MultiIndex.from_frame(Pi[["symbol", "date"]])
recovered = Pi[~key_i.isin(key_c)]
retained = Pi[key_i.isin(key_c)]
print(f"Recovered: {len(recovered):,} forecasts ({100*len(recovered)/len(Pi):.1f}% of the imputed test sample), "
      f"{recovered['symbol'].nunique():,} securities")
print(f"Median market value: recovered {recovered['MV'].median():.1f} mn, "
      f"retained {retained['MV'].median():.1f} mn\n")

rec = pd.DataFrame({
    "R2 recovered": annual_r2(recovered, MODELS).mean(),
    "R2 retained": annual_r2(retained, MODELS).mean(),
    "CS R2 recovered": pd.Series({m: 100 * r2_cross_section(recovered, m) for m in MODELS}),
    "CS R2 retained": pd.Series({m: 100 * r2_cross_section(retained, m) for m in MODELS})})
print("Models estimated on the imputed sample, 2010-2025 (%):")
print(rec.round(3).to_string())

if "n_imputed" in recovered.columns:
    bins = pd.cut(recovered["n_imputed"], [0, 1, 3, 10, 22], labels=["1", "2-3", "4-10", ">10"])
    print("\nelanet R^2 (%) on the recovered observations, by characteristics imputed:")
    for b, g in recovered.groupby(bins, observed=True):
        print(f"  {b:>5}: {100*r2_oos(g['target'], g['elanet']):>7.3f}  "
              f"cross-sectional {100*r2_cross_section(g, 'elanet'):>7.3f}  ({len(g):,} obs)")

## Diagnostics of the penalised models

The mixing parameter selected each year by `elanet`: ridge (0) points to a signal
spread across many predictors, lasso (1) to a concentrated one.

In [ ]:
sel = pd.DataFrame({s: R[s]["T"].query("model == 'elanet'").set_index("year")["rho"] for s in SAMPLES})
nz = pd.DataFrame({s: R[s]["T"].query("model == 'elanet'").set_index("year")["nonzero"] for s in SAMPLES})
print("elanet: mixing parameter / non-zero coefficients by year")
print(pd.concat({"rho": sel, "nonzero": nz}, axis=1).to_string())
for s in SAMPLES:
    e = sel[s]
    print(f"{s:14s} ridge {int((e==0).sum())} | enet {int((e==0.5).sum())} | lasso {int((e==1).sum())} years"
          f" | grid-edge hits {R[s]['T'].groupby('model')['at_boundary'].sum().to_dict()}")

In [ ]:
for s in SAMPLES:
    t = R[s]["T"].query("model == 'elanet'")
    names = R[s]["names"]
    C = np.vstack(t["coef"].values)
    share = (np.abs(C) > 1e-12).mean(axis=0)
    mean_c = C.mean(axis=0)
    order = np.lexsort((-np.abs(mean_c), -share))[:15]
    print(f"\n{s} -- elanet: fifteen most frequently retained predictors ({len(names)} in the full set)")
    for k in order:
        print(f"  {names[k]:40s} {share[k]:>5.0%}   {mean_c[k]:>+.5f}")

## Portfolio performance

Monthly deciles on the forecast, long-short top minus bottom, on unwinsorised returns
of the following month. Value weights use market capitalisation at formation.

In [ ]:
rows = []
for s in SAMPLES:
    for m in MODELS:
        ew, vw = decile_portfolios(R[s]["P"], m, ret="raw_target", weight="MV")
        for lab, port in [("equal", ew), ("value", vw)]:
            rows.append({"sample": s, "model": m, "weighting": lab, **portfolio_stats(port)})
perf = pd.DataFrame(rows)
print(perf.pivot_table(index="model", columns=["sample", "weighting"],
                       values=["mean_%", "t_stat"]).round(2).reindex(MODELS).to_string())

for s in SAMPLES:
    ew, vw = decile_portfolios(R[s]["P"], "elanet", ret="raw_target", weight="MV")
    print(f"\n{s} -- elanet, mean monthly excess return by decile (%):")
    print(pd.DataFrame({"equal": 100 * ew.mean(), "value": 100 * vw.mean()}).round(3).T.to_string())

## Save

In [ ]:
TAG = SAMPLES[0].replace("complete_case", "")   # "_mv25" for the main run, "" for the unrestricted extension
for s in SAMPLES:
    R[s]["P"].to_parquet(os.path.join(OUT_DIR, f"predictions_linear_{s}.parquet"), index=False)
    T = R[s]["T"]
    T.drop(columns="coef").to_csv(os.path.join(OUT_DIR, f"tuning_linear_{s}.csv"), index=False)
    co = pd.DataFrame(np.vstack(T["coef"].values), columns=R[s]["names"])
    co.insert(0, "model", T["model"].values); co.insert(1, "year", T["year"].values)
    co.to_parquet(os.path.join(OUT_DIR, f"coefficients_linear_{s}.parquet"), index=False)

for name, df in [("r2_avg_annual", avg), ("r2_pooled", pooled), ("r2_cross_section", cs),
                 ("r2_by_size", size_r2), ("predictive_slopes", ps_tab),
                 ("same_observations", same_obs), ("recovered", rec)]:
    df.to_csv(os.path.join(OUT_DIR, f"{name}_linear{TAG}.csv"))
perf.to_csv(os.path.join(OUT_DIR, f"portfolios_linear{TAG}.csv"), index=False)
for s in SAMPLES:
    dm[s].to_csv(os.path.join(OUT_DIR, f"diebold_mariano_linear_{s}.csv"))
print(f"Saved forecasts, tuning, coefficients and all result tables (suffix '{TAG}').")